# Clase 09 — Ejercicio intergrador Clínica MediCare

> Notebook de práctica. Ejecutá las celdas a medida que avanzás en el artículo.

La clínica MediCare lleva varios años registrando turnos médicos en un sistema interno. El sistema fue cambiando de versión con el tiempo, y los datos se fueron exportando a una base de datos SQLite sin demasiado control de calidad.

El director médico necesita un informe con algunas preguntas concretas sobre el funcionamiento de la clínica, pero antes de poder responderlas, alguien tiene que hacer la limpieza. Ese trabajo es el de ustedes, no se les dice qué problemas hay en los datos. Parte del ejercicio es encontrarlos.

> El archivo de la base de datos `medicare.db` ya está disponible para que lo uses en este ejercicio.

### 1) Exploración: ¿qué tiene esta base de datos?

Antes de tocar nada, el primer trabajo es entender qué hay adentro. Conectense a `medicare.db` y respondan estas preguntas.

1. ¿Qué tablas tiene la base de datos?
2. ¿Qué columnas y tipos tiene cada tabla?
3. ¿Cuántas filas tiene cada tabla?

In [2]:
import sqlite3
import pandas as pd

conexion = sqlite3.connect ("medicare.db")

#TABLAS DE LA BASE DE DATOS

df = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conexion
)
print(df)

        name
0  pacientes
1    medicos
2     turnos


In [3]:

#COLUMNAS Y TIPOS DE CADA TABLA
#CANTIDAD DE FILAS DE CADA TABLA
pacientes = pd.read_sql(
    "PRAGMA table_info (pacientes)", conexion
)
print(pacientes)
print(f"CANT COLUMNAS DE LA TABLA PACIENTES: {len(pacientes)}")
print(pacientes.shape)

medicos = pd.read_sql(
    "PRAGMA table_info (medicos)", conexion
)
print(medicos)
print(f"CANT COLUMNAS DE LA TABLA MEDICOS: {len(medicos)}")
print(medicos.shape)

turnos = pd.read_sql(
    "PRAGMA table_info (turnos)", conexion
)
print(turnos)
print(f"CANT COLUMNAS DE LA TABLA TURNOS: {len(turnos)}")
print(turnos.shape)

   cid              name     type  notnull dflt_value  pk
0    0                id  INTEGER        0       None   1
1    1            nombre     TEXT        0       None   0
2    2               dni     TEXT        0       None   0
3    3  fecha_nacimiento     TEXT        0       None   0
4    4            ciudad     TEXT        0       None   0
5    5       obra_social     TEXT        0       None   0
CANT COLUMNAS DE LA TABLA PACIENTES: 6
(6, 6)
   cid          name     type  notnull dflt_value  pk
0    0            id  INTEGER        0       None   1
1    1        nombre     TEXT        0       None   0
2    2  especialidad     TEXT        0       None   0
3    3     matricula     TEXT        0       None   0
4    4        activo     TEXT        0       None   0
CANT COLUMNAS DE LA TABLA MEDICOS: 5
(5, 6)
   cid         name     type  notnull dflt_value  pk
0    0           id  INTEGER        0       None   1
1    1  paciente_id  INTEGER        0       None   0
2    2    medico_id  

**Documenten sus hallazgos.** Antes de seguir al Paso 2, escriban en comentarios qué encontraron.

He podido identificar tres tablas: médicos, pacientes y turnos.
La tabla médicos compuesta por ID, NOMBRE, ESPECIALIDAD, MATRICULA Y ESTADO DE ACTIVIDAD. 
La tabla pacientes compuesta por ID, NOMBRE, DNI, FECHA DE NACIMIENTO, CIUDAD Y OBRA SOCIAL.
La tabla turnos compuesta por ID, ID PACIENTE, ID MEDICO, FECHA, HORA, MOTIVO, COSTO Y ESTADO.

### 2) Diagnóstico: ¿qué problemas tienen los datos?

Traigan las tres tablas a DataFrames y hagan un diagnóstico completo de cada una. Para cada tabla respondan:

- ¿Cuántos nulos hay por columna? ¿Qué porcentaje representan?
- ¿Hay filas duplicadas? ¿Cuáles?
- ¿Hay columnas con tipos incorrectos que van a impedir operar con los datos?
- ¿Hay valores que parecen errores aunque no sean nulos? (Pista: revisen la columna `costo` en `turnos`)


In [4]:

#COMO VER LA CANTIDAD DE REGISTROS DE LAS TABLAS

df_pacientes = pd.read_sql("SELECT * FROM pacientes", conexion)
print(df_pacientes.shape)

df_medicos = pd.read_sql("SELECT * FROM medicos", conexion)
print(df_medicos.shape)

df_turnos = pd.read_sql("SELECT * FROM turnos", conexion)
print(df_turnos.shape)

(66, 6)
(20, 5)
(210, 8)


In [5]:

print ("Medicos:")
#print(df_medicos.isnull().sum())
porcentaje_medicos_nulos = (df_medicos.isnull().sum() / len(df_medicos) * 100).round(1)
print(porcentaje_medicos_nulos)
print("----------------")

print("Pacientes:")
#print(df_pacientes.isnull().sum())
porcentaje_pacientes_nulos = (df_pacientes.isnull().sum() / len(df_pacientes) * 100).round(1)
print(porcentaje_pacientes_nulos)
print("----------------")

print("Turnos:")
#print(df_turnos.isnull().sum())
porcentaje_turnos_nulos = (df_turnos.isnull().sum() / len(df_turnos) * 100).round(1)
print(porcentaje_turnos_nulos)

Medicos:
id               0.0
nombre           0.0
especialidad     0.0
matricula       10.0
activo          20.0
dtype: float64
----------------
Pacientes:
id                   0.0
nombre               0.0
dni                 10.6
fecha_nacimiento    16.7
ciudad              12.1
obra_social         18.2
dtype: float64
----------------
Turnos:
id             0.0
paciente_id    0.0
medico_id      0.0
fecha          0.0
hora           7.6
motivo         7.6
costo          4.3
estado         0.0
dtype: float64


En la tabla MEDICOS se encuentran problemas de valores nulos tanto en la columna 'matrícula' como en la columna de 'actividad'.
En la tabla de PACIENTES se encuentran problemas de valores nulos en las tablas 'dni', 'fecha de nacimiento', 'ciudad' y 'obra social'.
En la tabla de TURNOS se encuentran problemas de valores nulos en las tablas de 'hora', 'motivo' y 'costo'.


La columna 'costo' en la tabla TURNOS es de tipo string: esto nos va a generar problemas en tanto que si queremos, por ejemplo, calcular promedios en el costo de las consultas, necesitamos trabajar con datos de tipo float, porque con los datos de tipo string no podemos hacer operaciones matemáticas.
Antes de hacer la conversión a float debemos hacer un tratamiento de los campos nulos.

In [6]:
print(df_medicos.duplicated(subset=["matricula"]).sum())
print(df_pacientes.duplicated(subset=["dni"]).sum())
print(df_turnos.duplicated(subset=["id"]).sum())


2
14
0


**Antes de seguir al Paso 3, escriban una lista con todos los problemas que encontraron, uno por línea.** No empiecen a limpiar todavía.

### 3) Decisiones: ¿qué hacemos con cada problema?

Este es el paso más importante del ejercicio. Para cada problema que encontraron en el Paso 2, decidan qué van a hacer y por qué. No hay una respuesta única correcta — lo que importa es que la decisión sea coherente con el análisis que necesitan hacer después.

Completen esta tabla antes de escribir código:

| Tabla | Columna | Problema | Decisión | Justificación |
|-------|---------|----------|----------|---------------|
| pacientes | ciudad | 2 nulos | Reemplazar el nulo con 'Sin datos' | No vale la pena borrar el registro  |
| pacientes | dni | 1 nulo | Borrar el registro | El DNI es un dato fundamental |
| Turnos | monto | Error de tipo | Convertir de str a float | Es necesario para realizar operaciones matemáticas |
| médicos | matrícula | duplicado | Borrar el duplicado | No es necesario conservar ambas |
| médicos | activo | 4 nulos | Reemplazas el nulo con 'Sin datos' | No vale la pena borrar el registro
| médicos | matrícula  | 2 nulos | Reemplazar el nulo con 'Sin datos' | No vale la pena borrar el registro
| médicos | nombre | duplicado | Borrar el duplicado | Genera registros no válidos



### 4) Limpieza: aplicar las decisiones

Ahora sí, implementen las decisiones del Paso 3. Recuerden:

- Primero eliminen duplicados
- Después manejen nulos
- Después conviertan tipos

In [28]:
# Limpieza de pacientes
df_pacientes_limpio = df_pacientes.copy()
df_pacientes_limpio = df_pacientes_limpio.drop_duplicates(subset=["dni"])
df_pacientes_limpio = df_pacientes_limpio.dropna(subset=["dni"])
df_pacientes_limpio["ciudad"]= df_pacientes_limpio["ciudad"].fillna("Sin datos")


# Limpieza de médicos
df_medicos_limpio = df_medicos.copy()
df_medicos_limpio = df_medicos_limpio.drop_duplicates(subset= ["matricula"])
df_medicos_limpio = df_medicos_limpio.drop_duplicates (subset=["nombre"])
df_medicos_limpio["activo"] = df_medicos_limpio["activo"].fillna("Sin datos")
df_medicos_limpio["matricula"]= df_medicos_limpio["matricula"].fillna("Sin datos")


# Limpieza de turnos
df_turnos_limpio = df_turnos.copy()
# Tu código acá
df_turnos_limpio["costo"]= pd.to_numeric(df_turnos_limpio["costo"], errors="coerce")


> **Por qué `.copy()`:** cuando hacen `df_pacientes_limpio = df_pacientes`, no están creando una copia — están apuntando al mismo objeto. Cualquier cambio en `df_pacientes_limpio` modifica también `df_pacientes`. `.copy()` crea una copia independiente, así conservan los datos originales para comparar.

Al terminar, impriman un resumen de qué cambió en cada tabla:

In [29]:
print(f"Filas pacientes originales:   {len(df_pacientes)}")
print(f"Filas DF pacientes limpio:     {len(df_pacientes_limpio)}")
print("--------")
print(f"Filas médicos originales:   {len(df_medicos)}")
print(f"Filas DF médicos limpio: {len(df_medicos_limpio)}")
print("--------")
print(f"Filas turno originales:   {len(df_turnos)}")
print(f"Filas DF turnos limpio: {len(df_turnos_limpio)}")

Filas pacientes originales:   66
Filas DF pacientes limpio:     51
--------
Filas médicos originales:   20
Filas DF médicos limpio: 12
--------
Filas turno originales:   210
Filas DF turnos limpio: 210


In [31]:
df_medicos_limpio.to_sql("df_medicos_limpio", conexion, if_exists="replace", index=False)
df_turnos_limpio.to_sql("df_turnos_limpio", conexion, if_exists="replace", index=False)
df_pacientes_limpio.to_sql("df_pacientes_limpio", conexion, if_exists="replace", index="False")

51

### 5) — Análisis: responder las preguntas del director

Con los datos limpios, respondan estas cuatro preguntas. Para cada una, combinen SQL (para traer los datos necesarios) con Pandas (para calcular o presentar el resultado):

**Pregunta 1:** ¿Cuántos turnos realizó cada médico? Mostrá el resultado ordenado de mayor a menor.

**Pregunta 2:** ¿Cuál es el costo total facturado por especialidad médica? Considerá solo los turnos con estado `'realizado'`.

**Pregunta 3:** ¿Qué obra social tiene más pacientes registrados? ¿Y cuántos pacientes no tienen obra social?

**Pregunta 4:** ¿Cuántos turnos fueron cancelados o marcados como `'ausente'`? ¿Qué médico tuvo más cancelaciones?


In [51]:
#Pregunta 1:
primera_query= pd.read_sql("""
                  SELECT m.id, m.nombre AS medico, COUNT(t.id) as cantidad_turnos 
                  FROM df_medicos_limpio m 
                  JOIN df_turnos_limpio as t ON m.id = t.medico_id
                  GROUP BY m.id
                  ORDER BY cantidad_turnos DESC""",
    conexion)

In [46]:
#Pregunta 2
segunda_query = pd.read_sql(""" SELECT m.especialidad, SUM(t.costo) AS costo_total
                  FROM df_medicos_limpio m
                  JOIN df_turnos_limpio t ON m.id = t.medico_id
                  WHERE t.estado like 'realizado'
                  GROUP BY m.especialidad
                  ORDER BY costo_total DESC
                  """, conexion)

In [48]:
#Pregunta 3
tercera_query = pd.read_sql("""SELECT obra_social, COUNT(id) AS cantidad_pacientes
                  FROM df_pacientes_limpio
                  WHERE obra_social IS NOT NULL AND obra_social NOT IN ('Sin datos')
                  GROUP BY obra_social
                  ORDER BY cantidad_pacientes DESC"""
                  , conexion)

cuarta_query = pd.read_sql("""SELECT COUNT(id) AS pacientes_sin_os
                  FROM df_pacientes_limpio
                  WHERE obra_social IS NULL
                  OR obra_social IN ('Sin datos')""", conexion)

In [47]:
#Pregunta 4
quinta_query = pd.read_sql("""SELECT estado, COUNT (id) AS cantidad
                  FROM df_turnos_limpio
                  WHERE estado IN ('ausente')
                  GROUP BY estado;
""", conexion)
sexta_query = pd.read_sql(""" SELECT m.nombre AS medico, COUNT(t.id) AS total_cancelaciones
                  FROM df_medicos_limpio m
                  JOIN df_turnos_limpio t on m.id = t.medico_id
                  WHERE t.estado IN ('ausente', 'cancelado')
                  GROUP BY m.nombre
                  ORDER BY total_cancelaciones DESC LIMIT 1
""", conexion)

### 6) Reporte final

Escriban un bloque de código que imprima un reporte con los resultados de los cuatro puntos anteriores, en formato legible. No es necesario que sea elaborado — lo importante es que quede claro qué encontraron.

In [58]:
print("=" * 50)
print("REPORTE CLÍNICA MEDICARE")
print("=" * 50)

print("Cantidad de turnos por médico: ")
print(primera_query)
print("-" * 50)
print("Total facturado por especialidad: ")
print(segunda_query)
print("-" * 50)
print("Cantidad de pacientes por obra social: ")
print(tercera_query)
print("-" * 50)
print("Cantidad de pacientes sin OS: ")
print(cuarta_query)
print("-" * 50)
print("Cantidad de turnos cancelados: ")
print(quinta_query)
print("-" * 50)
print("Médico con más cancelaciones: ")
print(sexta_query)

REPORTE CLÍNICA MEDICARE
Cantidad de turnos por médico: 
    id               medico  cantidad_turnos
0    1    Dra. Beatriz Sosa               20
1    6   Dr. Sebastián Ríos               18
2    3  Dra. Carmen Aguirre               14
3   20    Dra. Silvina Vega               13
4   14   Dra. Andrea Blanco               11
5   19   Dr. Ezequiel Bravo               10
6    7   Dra. Laura Giménez               10
7    2     Dr. Nicolás Vera               10
8   17     Dr. Hernán Costa                9
9   15     Dr. Martín Cufré                9
10   4     Dr. Pablo Méndez                9
11  16    Dra. Verónica Paz                5
--------------------------------------------------
Total facturado por especialidad: 
     especialidad  costo_total
0     Cardiología      80400.0
1       Pediatría      79600.0
2  Clínica Médica      63200.0
3    Dermatología      45200.0
4   Traumatología      42700.0
5     Ginecología      27300.0
6     Psiquiatría      21100.0
------------------------

→ [Ir a la solución](./solucion.md)